In [2]:
%load_ext autoreload
%autoreload 2

import torch
from torch import nn, Tensor
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.utils.data import Dataset, random_split
from torch.nn.utils.rnn import pad_sequence
import json
import random
import math
from torch.utils.tensorboard import SummaryWriter
import time
from utils.feature_engineering import get_delta_features, get_elapsed_feature
import numpy as np


In [5]:
from dataset.otto_trace import TraceOttoDataSet

#DataSet    
dataset_processed = TraceOttoDataSet(
    file_name='train.jsonl',
    input_seq_len=64,
    min_timestamps_per_sample=16,
    max_samples=1000
)

      
#See if the Lenght of the new inputs are the Lenght Sequence
for i,sample in enumerate(dataset_processed):
    len_sample = sample["inputs"]
    print(f"Len of the timestamps after the L sequence cut {len(len_sample["timestamps"])}")
    assert len(len_sample["timestamps"]) == 64 
    
    if i == 0:
        break    


print("================================================ (Logits part) ===================================================")
print("Logits Data_set_processed the ATC (Add to the Cart)")
print(dataset_processed.__ATC_task_logit__())
    
print("Logits for SAT4 (Seeing the same Aid 4 times)")
print(dataset_processed.__SAT__task_logit__())
    
print("Logits for PD1 (Make any Purchase within a day)")
print(dataset_processed.__PD1_task_logit___())
    
print("Logits for RA1 (Return to the same Aid in 1 days)")
print(dataset_processed.__RA1_task_logit___())

Len of the timestamps after the L sequence cut 64
================================================ (Logits part) ===================================================
Logits Data_set_processed the ATC (Add to the Cart)
[1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0,

In [6]:
# Data splitting train/test
train_portion = 0.80
train_data, test_data = random_split(dataset=dataset_processed, lengths=[train_portion, 1 - train_portion])


#DEV_SET
"""
validation_loader = DataLoader(
    dataset=val_data,
    batch_size=32,
    collate_fn=custom_collate,
    shuffle=False
)
"""

#TRAIN SET
train_loader = DataLoader(
    dataset=train_data,
    batch_size=32,
    shuffle=True
)
#TEST SET
test_loader = DataLoader(
    dataset=test_data,
    batch_size=32,
    shuffle=False
)


In [8]:
from model.trace import TRACE

max_aid = max(
    session["inputs"]["aid"].max().item()
    for session in dataset_processed
)
max_type = max(
    session["inputs"]["type"].max().item()
    for session in dataset_processed
)

num_embeddings_aid = max_aid + 1  
num_embeddings_event_type = max_type + 1
trace_model = TRACE(
    num_embeddings_aid=num_embeddings_aid,
    num_embeddings_event_type=num_embeddings_event_type,
    embedding_dim=32,
    num_classes=1 #4 # Jan: You are doing only one class here, so not 4 classes...
)

In [9]:
for batch_training in train_loader:
    sample = batch_training["inputs"]

    print(
        f"Shape Aids: {sample['aid'].shape}, "
        f"Shape Timestamps: {sample['timestamps'].shape}, "
        f"Shape Type: {sample['type'].shape}"
    )
    break  

Shape Aids: torch.Size([32, 64]), Shape Timestamps: torch.Size([32, 64]), Shape Type: torch.Size([32, 64])


In [ ]:
from utils.feature_engineering import get_delta_features, get_elapsed_feature

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(trace_model.parameters(), lr=1e-5, weight_decay=1e-6)

num_epochs = 15

#Summary Writer for tensorBoard
tensor_board_writer = SummaryWriter(log_dir=f"runs/exp_{time.time()}")
trace_model.train()
for epoch in range(num_epochs):
    # -------------------------------TRAINING ---------------------------
    trace_model.train()
    epoch_loss = 0.0

    correct_train = 0
    total_train = 0

    for batch_trainer in train_loader:
        evidence = batch_trainer["inputs"]
        label_train = batch_trainer["targets"]["ATC"].unsqueeze_(1)

        delta_elapsed = get_elapsed_feature(evidence["timestamps"])
        delta_between = get_delta_features(evidence["timestamps"])

        pred = trace_model(
            evidence["aid"],
            evidence["type"],
            delta_elapsed,
            delta_between
        )

        loss = criterion(pred, label_train.float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        probs = torch.sigmoid(pred)
        preds = (probs >= 0.5).float()

        correct_train += (preds == label_train).sum().item()
        total_train += label_train.numel()

    train_loss = epoch_loss / len(train_loader)
    train_acc = correct_train / max(total_train, 1)
    
    #TensorBoard Writer
    tensor_board_writer.add_scalar("Training/Loss", val_loss, epoch)
    tensor_board_writer.add_scalar("Training/Accuracy", val_acc, epoch)
    
    # ------------------------------- VALIDATION ---------------------------

    trace_model.eval()
    val_loss = 0.0
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for batch_test in test_loader:
            evidence = batch_test["inputs"]
            label_test = batch_test["targets"]["ATC"].unsqueeze_(1)

            delta_elapsed = get_elapsed_feature(evidence["timestamps"])
            delta_between = get_delta_features(evidence["timestamps"])

            pred_test = trace_model(
                evidence["aid"],
                evidence["type"],
                delta_elapsed,
                delta_between
            )

            loss_v = criterion(pred_test, label_test.float())
            val_loss += loss_v.item()

            probs_test = torch.sigmoid(pred_test)
            preds_test = (probs_test >= 0.5).float()

            correct_test += (preds_test == label_test).sum().item()
            total_test += label_test.numel()

    val_loss /= len(test_loader)
    val_acc = correct_test / max(total_test, 1)
    
    #TensorBoard Writer
    tensor_board_writer.add_scalar("Val/Loss", val_loss, epoch)
    tensor_board_writer.add_scalar("Val/Accuracy", val_acc, epoch)
    
    
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )


Epoch [1/15] Train Loss: 0.6734 | Train Acc: 0.7486 | Val Loss: 0.6753 | Val Acc: 0.7007
Epoch [2/15] Train Loss: 0.6698 | Train Acc: 0.7596 | Val Loss: 0.6710 | Val Acc: 0.7153
Epoch [3/15] Train Loss: 0.6643 | Train Acc: 0.7614 | Val Loss: 0.6672 | Val Acc: 0.7153
Epoch [4/15] Train Loss: 0.6588 | Train Acc: 0.7632 | Val Loss: 0.6634 | Val Acc: 0.7153
Epoch [5/15] Train Loss: 0.6540 | Train Acc: 0.7614 | Val Loss: 0.6598 | Val Acc: 0.7153
Epoch [6/15] Train Loss: 0.6515 | Train Acc: 0.7614 | Val Loss: 0.6562 | Val Acc: 0.7153
Epoch [7/15] Train Loss: 0.6450 | Train Acc: 0.7614 | Val Loss: 0.6528 | Val Acc: 0.7153
Epoch [8/15] Train Loss: 0.6412 | Train Acc: 0.7614 | Val Loss: 0.6495 | Val Acc: 0.7153
Epoch [9/15] Train Loss: 0.6419 | Train Acc: 0.7614 | Val Loss: 0.6463 | Val Acc: 0.7153
Epoch [10/15] Train Loss: 0.6409 | Train Acc: 0.7614 | Val Loss: 0.6434 | Val Acc: 0.7153
Epoch [11/15] Train Loss: 0.6347 | Train Acc: 0.7614 | Val Loss: 0.6407 | Val Acc: 0.7153
Epoch [12/15] Train